# Modelo No Supervisado: Segmentacion de Clientes (Clustering)
## Proyecto Final - Data Science YPF

- Se aplica `KMeans` sobre variables de comportamiento crediticio para encontrar perfiles de clientes similares, **sin usar la etiqueta** `SeriousDlqin2yrs`.
- Al final se compara cada cluster con la tasa real de mora, solo como validacion: la etiqueta no se usa para entrenar ni para elegir la cantidad de clusters.
- KMeans es muy sensible a los valores extremos y a la escala de las variables, por lo que el preprocesamiento es una parte central de este notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

ModuleNotFoundError: No module named 'pandas'

## 1. Carga y preparacion de datos

Se parte del dataset limpio del notebook 01. Para el clustering se aplican cuatro pasos:

1. **Se separan los 269 clientes con codigo 96/98** en las variables de atraso. Sus atrasos reales son desconocidos (estan como NaN), y si se imputaran con la mediana (0) quedarian mezclados con los clientes sin atrasos, aunque tienen 54.6% de mora. Se analizan aparte como un segmento propio.
2. **Se imputan los faltantes con la mediana.** KMeans no acepta NaN. Aca no hay riesgo de data leakage porque el clustering no tiene set de test: no se evalua sobre datos nuevos.
3. **Se aplica logaritmo (`log1p`) a las variables con cola larga** (ingreso y conteos). KMeans usa distancias, y sin esta transformacion unos pocos valores muy altos (por ejemplo, ingresos de millones) dominarian la formacion de los grupos.
4. **Se estandarizan todas las variables** con `StandardScaler`, para que todas pesen lo mismo en la distancia.

In [ ]:
df = pd.read_csv("../dataset/cs-training-clean.csv")

features_clustering = [
    "RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio", "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines", "NumberOfDependents",
]

# 1. Clientes con codigo 96/98: se analizan aparte
especiales = df[df["atraso_codigo_especial"] == 1].copy()
df_cluster = df[df["atraso_codigo_especial"] == 0].reset_index(drop=True)
print("Clientes para el clustering:", len(df_cluster), "| Clientes con codigo 96/98 (aparte):", len(especiales))

# 2. Imputacion de faltantes con la mediana
X = df_cluster[features_clustering].copy()
X = X.fillna(X.median())

# 3. Logaritmo en las variables con cola larga
columnas_log = ["MonthlyIncome", "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate",
                "NumberRealEstateLoansOrLines", "NumberOfDependents"]
X_transformado = X.copy()
X_transformado[columnas_log] = np.log1p(X_transformado[columnas_log])

# 4. Estandarizacion
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformado)
X_scaled.shape

**Resultado:** el clustering se aplica sobre 149.731 clientes y 8 variables. Los 269 clientes con codigo 96/98 quedan fuera y se analizan en la seccion 5.

## 2. Eleccion de k

Se corre KMeans para k entre 2 y 8 sobre una muestra de 10.000 clientes (el silhouette compara todos los pares de puntos, y con 150.000 seria muy lento) y se calculan cuatro indicadores:

- **Inercia (metodo del codo):** distancia total de cada cliente al centro de su cluster. Siempre baja al aumentar k; se busca el punto donde deja de bajar rapido.
- **Silhouette:** que tan cerca esta cada cliente de su propio cluster en comparacion con el cluster vecino. Va de -1 a 1; mas alto es mejor. Como referencia: mayor a 0.5 indica estructura fuerte, entre 0.25 y 0.5 debil, y menor a 0.25 casi sin estructura.
- **Davies-Bouldin:** compara la dispersion dentro de cada cluster con la distancia entre clusters. Mas bajo es mejor.
- **Calinski-Harabasz:** cociente entre la dispersion entre clusters y dentro de ellos. Mas alto es mejor; solo sirve para comparar valores de k en el mismo dataset.

In [ ]:
muestra = X_scaled[np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), 10000, replace=False)]

metricas_k = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(muestra)
    metricas_k.append({
        "k": k,
        "inercia": km.inertia_,
        "silhouette": silhouette_score(muestra, labels),
        "davies_bouldin": davies_bouldin_score(muestra, labels),
        "calinski_harabasz": calinski_harabasz_score(muestra, labels),
    })

tabla_k = pd.DataFrame(metricas_k).set_index("k")
display(tabla_k.round(3))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
graficos = [
    ("inercia", "Metodo del codo (inercia)", "tab:blue"),
    ("silhouette", "Silhouette (mas alto es mejor)", "darkorange"),
    ("davies_bouldin", "Davies-Bouldin (mas bajo es mejor)", "green"),
    ("calinski_harabasz", "Calinski-Harabasz (mas alto es mejor)", "purple"),
]
for ax, (col, titulo, color) in zip(axes.flatten(), graficos):
    ax.plot(tabla_k.index, tabla_k[col], marker="o", color=color)
    ax.set_title(titulo)
    ax.set_xlabel("k")
plt.tight_layout()
plt.show()

**Hallazgo:** ningun valor de k se destaca con claridad, y las metricas no coinciden entre si:

- El **silhouette** queda entre 0.18 y 0.21 para todos los k: los clientes no forman grupos naturales bien separados. Es habitual en datos de comportamiento, donde los clientes varian de forma continua.
- El silhouette y el Davies-Bouldin prefieren k = 7 (0.209 y 1.266); el Calinski-Harabasz prefiere k = 6 (2.017). La curva del codo baja de forma pareja, sin un quiebre claro.

Por eso **k no se elige por las metricas sino por interpretabilidad**. 

Se elige k = 4 porque da cuatro segmentos con perfiles faciles de describir y con niveles de riesgo bien diferenciados. 
Con k = 3 se pierde la diferencia entre los clientes mayores de bajo riesgo y los jovenes de riesgo medio. 
Con k = 5 aparece un grupo extra de unos 2.250 clientes definido por tener ingreso 0, una particularidad de los datos mas que un perfil de comportamiento.

La segmentacion se entiende como una herramienta de negocio para dividir a los clientes en perfiles, no como el descubrimiento de grupos que "existen" naturalmente en los datos.

## 2 bis. Sensibilidad a valores extremos y estabilidad

Para cuantificar el efecto del tratamiento se comparan dos representaciones sobre los mismos clientes:

- **Sin tratamiento:** variables crudas, imputación global solo para poder ejecutar KMeans y estandarización, sin `log1p` ni reglas para outliers.
- **Tratada:** reglas del notebook 01, imputación, `log1p` y estandarización.

La comparación se hace con `k = 4`, y se reportan métricas internas, tamaños de clusters y el porcentaje de clientes cuya asignación cambia. También se repite KMeans con varias semillas para medir estabilidad mediante Adjusted Rand Index (ARI).

In [ ]:
from sklearn.metrics import adjusted_rand_score

df_raw = pd.read_csv("../dataset/cs-training.csv")
columnas_atraso_raw = ["NumberOfTime30-59DaysPastDueNotWorse",
                       "NumberOfTime60-89DaysPastDueNotWorse",
                       "NumberOfTimes90DaysLate"]
mask_validos = df_raw[columnas_atraso_raw].lt(96).all(axis=1)
df_raw_cluster = df_raw.loc[mask_validos].reset_index(drop=True)

# Representacion sin tratamiento de outliers: solo imputacion necesaria y escala.
X_raw = df_raw_cluster[features_clustering].copy()
X_raw = X_raw.fillna(X_raw.median())
X_raw_scaled = StandardScaler().fit_transform(X_raw)

muestra_idx_sens = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), 10000, replace=False)
km_raw = KMeans(n_clusters=K_ELEGIDO, random_state=RANDOM_STATE, n_init=10)
km_tratado = KMeans(n_clusters=K_ELEGIDO, random_state=RANDOM_STATE, n_init=10)
labels_raw = km_raw.fit_predict(X_raw_scaled)
labels_tratado = km_tratado.fit_predict(X_scaled)

def metricas_clustering(X_eval, labels):
    return {
        "silhouette": silhouette_score(X_eval, labels),
        "davies_bouldin": davies_bouldin_score(X_eval, labels),
        "calinski_harabasz": calinski_harabasz_score(X_eval, labels),
    }

comparacion_kmeans = pd.DataFrame({
    "sin_tratamiento": metricas_clustering(X_raw_scaled[muestra_idx_sens], labels_raw[muestra_idx_sens]),
    "tratado": metricas_clustering(X_scaled[muestra_idx_sens], labels_tratado[muestra_idx_sens]),
}).T
comparacion_kmeans["clientes"] = [len(labels_raw), len(labels_tratado)]
comparacion_kmeans["cambio_asignacion_%"] = [np.nan, (labels_raw != labels_tratado).mean() * 100]
display(comparacion_kmeans.round(3))
display(pd.DataFrame({
    "sin_tratamiento": pd.Series(labels_raw).value_counts().sort_index(),
    "tratado": pd.Series(labels_tratado).value_counts().sort_index(),
}).fillna(0).astype(int))

# Estabilidad: ARI contra la primera corrida, usando la misma muestra y varias semillas.
semillas = [0, 7, 21, 42, 99]
labels_estabilidad = []
for semilla in semillas:
    km = KMeans(n_clusters=K_ELEGIDO, random_state=semilla, n_init=10)
    labels_estabilidad.append(km.fit_predict(X_scaled[muestra_idx_sens]))
ari_estabilidad = [adjusted_rand_score(labels_estabilidad[0], labels) for labels in labels_estabilidad[1:]]
print(f"Cambio de asignacion entre representaciones: {comparacion_kmeans.loc['tratado', 'cambio_asignacion_%']:.1f}%")
print(f"ARI promedio entre semillas (tratado): {np.mean(ari_estabilidad):.3f}")
print(f"ARI minimo entre semillas (tratado): {np.min(ari_estabilidad):.3f}")

In [ ]:
K_ELEGIDO = 4
kmeans_final = KMeans(n_clusters=K_ELEGIDO, random_state=RANDOM_STATE, n_init=10)
df_cluster["cluster"] = kmeans_final.fit_predict(X_scaled)
df_cluster["cluster"].value_counts().sort_index()

## 3. Perfil de cada cluster

El perfil se calcula con las variables en sus unidades originales (despues de imputar, antes del logaritmo y de la estandarizacion), para que se pueda leer directamente. Se usa el promedio de cada variable, salvo para el ingreso, donde se usa la mediana porque unos pocos ingresos muy altos distorsionan el promedio. Los clusters se ordenan de menor a mayor tasa de mora.

In [ ]:
X_perfil = X.assign(cluster=df_cluster["cluster"])

perfil = X_perfil.groupby("cluster").mean()
perfil["MonthlyIncome"] = X_perfil.groupby("cluster")["MonthlyIncome"].median()
perfil = perfil.rename(columns={"MonthlyIncome": "MonthlyIncome (mediana)"})
perfil["clientes"] = df_cluster["cluster"].value_counts()
perfil["participacion_%"] = perfil["clientes"] / len(df_cluster) * 100
perfil["tasa_mora_%"] = df_cluster.groupby("cluster")["SeriousDlqin2yrs"].mean() * 100
perfil = perfil.sort_values("tasa_mora_%")
display(perfil.round(2).T)

**Hallazgo:** los cuatro segmentos, de menor a mayor riesgo:

| Segmento | Clientes | Perfil | Mora real |
|---|---|---|---|
| Mayores con poco uso de credito | 51.393 (34%) | Edad promedio 65 anos, usan el 12% de su limite, casi sin dependientes | 1.6% |
| Ingresos altos y mas productos | 56.995 (38%) | 49 anos, mayor ingreso (mediana 6.244), 11 lineas de credito y ~2 prestamos inmobiliarios | 5.6% |
| Jovenes con uso medio | 33.512 (22%) | 40 anos, usan el 53% de su limite, menor ingreso (mediana 4.154) y pocos productos | 7.7% |
| Con atrasos graves previos | 7.831 (5%) | 46 anos, usan el 78% de su limite y tienen 1.7 atrasos de 90+ dias en promedio | 41.7% |

El numero de cada cluster (0, 1, 2, 3) es solo una etiqueta: lo que importa es el perfil.

## 4. Visualizacion con PCA

PCA resume las 8 variables en 2 componentes para poder graficar a los clientes en un plano. Es solo una ayuda visual: si las 2 componentes explican una parte chica de la variabilidad, clusters que se ven superpuestos pueden estar separados en otras dimensiones.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df_cluster["cluster"], palette="tab10", s=6, alpha=0.4)
plt.title("Clusters proyectados en 2 componentes principales")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

print("Varianza explicada por las 2 componentes:", pca.explained_variance_ratio_.sum().round(3))

In [ ]:
pesos_pca = pd.DataFrame(pca.components_.T, index=features_clustering, columns=["PC1", "PC2"])
pesos_pca.sort_values("PC1", ascending=False).round(2)

**Hallazgo:** 
- Cada componente es una combinacion de las 8 variables. PC1 se asocia sobre todo a la cantidad de productos de credito (lineas abiertas y prestamos inmobiliarios), y PC2 a la presion financiera: sube con la utilizacion del credito, los dependientes, el endeudamiento y los atrasos, y baja con la edad.
- Las 2 componentes explican el 43.8% de la variabilidad, menos de la mitad. 
- El grafico muestra solo una parte de la estructura, y es esperable que los clusters aparezcan superpuestos, en linea con el silhouette bajo: los segmentos dividen un continuo de clientes en zonas, no grupos aislados.

## 5. Validacion contra la mora real

El clustering no uso `SeriousDlqin2yrs`. Cruzarlo ahora permite responder si los grupos formados solo por comportamiento crediticio coinciden con perfiles de riesgo distintos. Se agrega como quinto grupo a los clientes con codigo 96/98, que se dejaron fuera del clustering.

In [ ]:
mora_segmentos = df_cluster.groupby("cluster")["SeriousDlqin2yrs"].agg(["count", "mean"])
mora_segmentos.index = [f"Cluster {c}" for c in mora_segmentos.index]
mora_segmentos.loc["Codigo 96/98 (fuera del clustering)"] = [len(especiales), especiales["SeriousDlqin2yrs"].mean()]
mora_segmentos.columns = ["clientes", "tasa_mora"]
mora_segmentos["clientes"] = mora_segmentos["clientes"].astype(int)
mora_segmentos["tasa_mora_%"] = (mora_segmentos["tasa_mora"] * 100).round(1)
mora_segmentos = mora_segmentos.drop(columns="tasa_mora").sort_values("tasa_mora_%")
display(mora_segmentos)

tasa_base = df["SeriousDlqin2yrs"].mean() * 100
ax = mora_segmentos["tasa_mora_%"].plot(kind="barh", figsize=(8, 4), title="Tasa real de mora por segmento")
ax.axvline(tasa_base, color="red", linestyle="--", label=f"Tasa promedio ({tasa_base:.1f}%)")
ax.set_xlabel("% de clientes con mora seria")
ax.legend()
plt.tight_layout()
plt.show()

**Hallazgo:** aunque KMeans no vio la etiqueta, los segmentos tienen niveles de riesgo muy distintos, contra una tasa promedio de 6.7%:

- **Mayores con poco uso de credito: 1.6%**, cuatro veces menos que el promedio.
- **Ingresos altos y mas productos: 5.6%** y **jovenes con uso medio: 7.7%**, cerca del promedio.
- **Con atrasos graves previos: 41.7%**, seis veces el promedio.
- **Codigo 96/98: 54.6%**, el grupo de mayor riesgo, analizado aparte.

El segmento de mayor riesgo esta definido casi por completo por una variable: el 99.8% de sus clientes tiene al menos un atraso de 90+ dias, y el 97% de los clientes con ese tipo de atraso cae en este segmento. Esto coincide con el modelo supervisado, donde el historial de atrasos esta entre las variables mas importantes.

## 6. Conclusiones

- Se identificaron 4 segmentos de clientes a partir de su comportamiento crediticio, sin usar la variable objetivo: mayores con poco uso de credito, clientes con ingresos altos y mas productos, jovenes con uso medio y clientes con atrasos graves previos.
- Las metricas de calidad (silhouette entre 0.18 y 0.21) indican que no hay grupos naturales bien separados. Ninguna metrica elige k = 4: se eligio por interpretabilidad, y la segmentacion se usa como herramienta de negocio.
- Al cruzar con la mora real, los segmentos tienen riesgos muy distintos: desde 1.6% hasta 41.7%, contra un promedio de 6.7%. La segmentacion, sin ver la etiqueta, separa a los clientes de bajo y alto riesgo.
- El segmento de mayor riesgo esta definido por el historial de atrasos de 90+ dias, en linea con el modelo supervisado.
- Los clientes con codigo 96/98 (54.6% de mora) son el grupo de mayor riesgo y conviene tratarlos como un segmento propio.
- Uso posible: politicas diferenciadas por perfil (limites, tasas, seguimiento), con foco en los segmentos de atrasos graves y codigo 96/98, y condiciones preferenciales para el segmento de bajo riesgo.